# Homework: Build a Purchasing Star and Link It to Sales

**Course:** Data Pipelines with Databricks and Fabric  
**Student:** Ahmet Okcu  
**Dataset:** Microsoft AdventureWorks2022 — Purchasing  
**Catalog:** `adventure_works`

**Medallion pipeline:** Bronze (raw staging) → Silver (named, typed, cleaned) → Gold (fact_purchases)

**Result:** Two stars sharing `dim_product`:
- Sales star: `fact_sales` + `dim_product` + `dim_sales_territory`
- Purchasing star: `fact_purchases` + `dim_vendor` (+ shared `dim_product`)

In [0]:
# Inspect the raw CSV files before loading: delimiter, column count, first data row
# The Lesson 1 CSVs were tab-delimited with no header row — verify the purchasing CSVs match
for csv_file in ["PurchaseOrderDetail.csv", "PurchaseOrderHeader.csv", "Vendor.csv"]:
    path = f"/Volumes/adventure_works/default/raw_data/{csv_file}"
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().rstrip("\n\r")
    tab_count = first_line.count("\t")
    comma_count = first_line.count(",")
    print(f"=== {csv_file} ===")
    print(f"  delimiter: TAB ({tab_count} tabs) | COMMA ({comma_count} commas)")
    print(f"  first row preview: {first_line[:150]}")
    print()

=== PurchaseOrderDetail.csv ===
  delimiter: TAB (10 tabs) | COMMA (0 commas)
  first row preview: 1	1	2022-04-29 00:00:00.000	4	1	50.2600	201.0400	3.00	.00	3.00	2022-04-22 00:00:00.000

=== PurchaseOrderHeader.csv ===
  delimiter: TAB (12 tabs) | COMMA (0 commas)
  first row preview: 1	5	4	258	1580	3	2022-04-15 00:00:00.000	2022-04-24 00:00:00.000	201.0400	16.0832	5.0260	222.1492	2022-04-24 00:00:00.000

=== Vendor.csv ===
  delimiter: TAB (7 tabs) | COMMA (0 commas)
  first row preview: 1492	AUSTRALI0001	Australia Bike Retailer	1	1	1		2022-12-22 00:00:00.000



In [0]:
# Look up the real column names and types in the CREATE TABLE statements of instawdb.sql
# (Microsoft's SQL Server install script — the single source of truth for the schema)

with open("/Volumes/adventure_works/default/raw_data/instawdb.sql", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

for table_name in ["PurchaseOrderDetail", "PurchaseOrderHeader", "Vendor"]:
    marker = f"CREATE TABLE [Purchasing].[{table_name}]"
    for i, line in enumerate(lines):
        if marker in line:
            print(f"===== {table_name} (instawdb.sql line {i + 1}) =====")
            j = i + 1
            while j < len(lines):
                col_line = lines[j].strip()
                if col_line.startswith("["):  # a column definition line
                    # drop inline CONSTRAINT / AS (computed) parts, keep name + type + nullability
                    clean = col_line.split("CONSTRAINT")[0].split(" AS ")[0].strip().rstrip(",")
                    print(f"  {clean}")
                if col_line.startswith(")"):  # end of the CREATE TABLE block
                    break
                j += 1
            print()
            break

===== PurchaseOrderDetail (instawdb.sql line 1454) =====
  [PurchaseOrderID] [int] NOT NULL
  [PurchaseOrderDetailID] [int] IDENTITY (1, 1) NOT NULL
  [DueDate] [datetime] NOT NULL
  [OrderQty] [smallint] NOT NULL
  [ProductID] [int] NOT NULL
  [UnitPrice] [money] NOT NULL
  [LineTotal]
  [ReceivedQty] [decimal](8, 2) NOT NULL
  [RejectedQty] [decimal](8, 2) NOT NULL
  [StockedQty]
  [ModifiedDate] [datetime] NOT NULL

===== PurchaseOrderHeader (instawdb.sql line 1473) =====
  [PurchaseOrderID] [int] IDENTITY (1, 1) NOT NULL
  [RevisionNumber] [tinyint] NOT NULL
  [Status] [tinyint] NOT NULL
  [EmployeeID] [int] NOT NULL
  [VendorID] [int] NOT NULL
  [ShipMethodID] [int] NOT NULL
  [OrderDate] [datetime] NOT NULL
  [ShipDate] [datetime] NULL
  [SubTotal] [money] NOT NULL
  [TaxAmt] [money] NOT NULL
  [Freight] [money] NOT NULL
  [TotalDue]
  [ModifiedDate] [datetime] NOT NULL

===== Vendor (instawdb.sql line 1756) =====
  [BusinessEntityID] [int] NOT NULL
  [AccountNumber] [AccountNumb

## Bronze Layer — Raw Staging Tables

**What:** Load the 3 raw CSVs exactly as they are, as 3 separate staging tables.

**Why:** Bronze captures raw data untouched. Because the CSVs have **no header row** and we read with `header=False`, Spark assigns generic column names — `_c0`, `_c1`, `_c2`, ... — exactly what the homework expects. The real names are applied in Silver.

**Pattern:** `read CSV (tab-delimited) → saveAsTable (adventure_works.bronze)`

In [0]:
# BRONZE — load the 3 raw CSVs as staging tables
# header=False  -> columns come in as _c0, _c1, _c2 ... (CSVs have no header row)
# sep='\t'      -> the CSV extracts are tab-delimited
# No schema/inference -> every column stays STRING: truly raw, untouched data

bronze_sources = {
    "PurchaseOrderDetail.csv": "stg_purchase_order_detail",
    "PurchaseOrderHeader.csv": "stg_purchase_order_header",
    "Vendor.csv": "dim_vendor",
}

for csv_file, table_name in bronze_sources.items():
    df = (
        spark.read
        .option("header", False)
        .option("sep", "\t")
        .csv(f"/Volumes/adventure_works/default/raw_data/{csv_file}")
    )
    df.write.mode("overwrite").saveAsTable(f"adventure_works.bronze.{table_name}")
    print(f"{csv_file} -> adventure_works.bronze.{table_name}: {df.count()} rows")

PurchaseOrderDetail.csv -> adventure_works.bronze.stg_purchase_order_detail: 8845 rows
PurchaseOrderHeader.csv -> adventure_works.bronze.stg_purchase_order_header: 4012 rows
Vendor.csv -> adventure_works.bronze.dim_vendor: 104 rows


In [0]:
# BRONZE verification — confirm the generic _c0, _c1 ... column names and row counts
for table_name in ["stg_purchase_order_detail", "stg_purchase_order_header", "dim_vendor"]:
    print(f"===== adventure_works.bronze.{table_name} =====")
    display(spark.sql(f"DESCRIBE adventure_works.bronze.{table_name}"))

===== adventure_works.bronze.stg_purchase_order_detail =====


col_name,data_type,comment
_c0,string,null
_c1,string,null
_c2,string,null
_c3,string,null
_c4,string,null
_c5,string,null
_c6,string,null
_c7,string,null
_c8,string,null
_c9,string,null


===== adventure_works.bronze.stg_purchase_order_header =====


col_name,data_type,comment
_c0,string,null
_c1,string,null
_c2,string,null
_c3,string,null
_c4,string,null
_c5,string,null
_c6,string,null
_c7,string,null
_c8,string,null
_c9,string,null


===== adventure_works.bronze.dim_vendor =====


col_name,data_type,comment
_c0,string,null
_c1,string,null
_c2,string,null
_c3,string,null
_c4,string,null
_c5,string,null
_c6,string,null
_c7,string,null


## Silver Layer — Real Names, Real Types, Data Quality

**What:** Take the 3 Bronze staging tables, apply the real column names from `instawdb.sql`, cast SQL Server types to Databricks types, and enforce a data-quality rule.

**Type mapping (SQL Server → Databricks):**  
| SQL Server | Databricks |
| --- | --- |
| int | INT |
| smallint / tinyint | SMALLINT |
| money | DECIMAL(19,4) |
| decimal(8,2) | DECIMAL(8,2) |
| datetime | TIMESTAMP |
| Flag (bit, 0/1) | INT |
| AccountNumber / Name / nvarchar | STRING |

**Data-quality rule (stg_purchase_order_detail):** drop rows with a NULL `ProductID` or a negative `LineTotal`.

In [0]:
# SILVER 1/2 — stg_purchase_order_detail: real column names + types + data-quality filter
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.silver")

# Column list and types straight from the CREATE TABLE [Purchasing].[PurchaseOrderDetail] statement
# (_c0, _c1, ... map to these names in order)
detail_columns = [
    ("PurchaseOrderID", "int"), ("PurchaseOrderDetailID", "int"), ("DueDate", "timestamp"),
    ("OrderQty", "smallint"), ("ProductID", "int"), ("UnitPrice", "decimal(19,4)"),
    ("LineTotal", "decimal(19,4)"), ("ReceivedQty", "decimal(8,2)"),
    ("RejectedQty", "decimal(8,2)"), ("StockedQty", "decimal(8,2)"), ("ModifiedDate", "timestamp"),
]

df = spark.table("adventure_works.bronze.stg_purchase_order_detail")
df = df.toDF(*[c for c, _ in detail_columns])                          # _c0.._c10 -> real names
df = df.withColumns({c: F.col(c).cast(t) for c, t in detail_columns})  # STRING -> real types, one pass

# Data-quality rule: drop rows with NULL ProductID or negative LineTotal
rows_before = df.count()
df = df.filter(F.col("ProductID").isNotNull() & (F.col("LineTotal") >= 0))
rows_after = df.count()

df.write.mode("overwrite").saveAsTable("adventure_works.silver.stg_purchase_order_detail")
print(f"stg_purchase_order_detail: {rows_before} -> {rows_after} rows "
      f"(dropped {rows_before - rows_after} rows: NULL ProductID or negative LineTotal)")

stg_purchase_order_detail: 8845 -> 8845 rows (dropped 0 rows: NULL ProductID or negative LineTotal)


In [0]:
# SILVER 2/2 — stg_purchase_order_header and dim_vendor: real column names + types
from pyspark.sql import functions as F

header_columns = [  # from CREATE TABLE [Purchasing].[PurchaseOrderHeader]
    ("PurchaseOrderID", "int"), ("RevisionNumber", "smallint"), ("Status", "smallint"),
    ("EmployeeID", "int"), ("VendorID", "int"), ("ShipMethodID", "int"),
    ("OrderDate", "timestamp"), ("ShipDate", "timestamp"), ("SubTotal", "decimal(19,4)"),
    ("TaxAmt", "decimal(19,4)"), ("Freight", "decimal(19,4)"), ("TotalDue", "decimal(19,4)"),
    ("ModifiedDate", "timestamp"),
]
vendor_columns = [  # from CREATE TABLE [Purchasing].[Vendor]
    ("BusinessEntityID", "int"), ("AccountNumber", "string"), ("Name", "string"),
    ("CreditRating", "smallint"), ("PreferredVendorStatus", "int"), ("ActiveFlag", "int"),
    ("PurchasingWebServiceURL", "string"), ("ModifiedDate", "timestamp"),
]

for table_name, columns in [
    ("stg_purchase_order_header", header_columns),
    ("dim_vendor", vendor_columns),
]:
    df = spark.table(f"adventure_works.bronze.{table_name}")
    df = df.toDF(*[c for c, _ in columns])
    df = df.withColumns({c: F.col(c).cast(t) for c, t in columns})
    df.write.mode("overwrite").saveAsTable(f"adventure_works.silver.{table_name}")
    print(f"adventure_works.silver.{table_name}: {df.count()} rows, {len(columns)} columns")

adventure_works.silver.stg_purchase_order_header: 4012 rows, 13 columns
adventure_works.silver.dim_vendor: 104 rows, 8 columns


In [0]:
# SILVER verification — real names and real types, no _c0 columns left
for table_name in ["stg_purchase_order_detail", "stg_purchase_order_header", "dim_vendor"]:
    print(f"===== adventure_works.silver.{table_name} =====")
    display(spark.sql(f"DESCRIBE adventure_works.silver.{table_name}"))

===== adventure_works.silver.stg_purchase_order_detail =====


col_name,data_type,comment
PurchaseOrderID,int,null
PurchaseOrderDetailID,int,null
DueDate,timestamp,null
OrderQty,smallint,null
ProductID,int,null
UnitPrice,"decimal(19,4)",null
LineTotal,"decimal(19,4)",null
ReceivedQty,"decimal(8,2)",null
RejectedQty,"decimal(8,2)",null
StockedQty,"decimal(8,2)",null


===== adventure_works.silver.stg_purchase_order_header =====


col_name,data_type,comment
PurchaseOrderID,int,null
RevisionNumber,smallint,null
Status,smallint,null
EmployeeID,int,null
VendorID,int,null
ShipMethodID,int,null
OrderDate,timestamp,null
ShipDate,timestamp,null
SubTotal,"decimal(19,4)",null
TaxAmt,"decimal(19,4)",null


===== adventure_works.silver.dim_vendor =====


col_name,data_type,comment
BusinessEntityID,int,null
AccountNumber,string,null
Name,string,null
CreditRating,smallint,null
PreferredVendorStatus,int,null
ActiveFlag,int,null
PurchasingWebServiceURL,string,null
ModifiedDate,timestamp,null
# Clustering Information,,
# col_name,data_type,comment


## Gold Layer — fact_purchases

**What:** Join `stg_purchase_order_detail` to `stg_purchase_order_header` on `PurchaseOrderID` to pull in `VendorID` and `OrderDate`, one row per PO line. Save as the single fact table `fact_purchases`, next to the Silver `dim_vendor`.

**Why:** The detail rows know *what* was bought (product, qty, price) but not *who from* or *when it was ordered* — that lives in the header. The join completes the purchasing star:

- **Purchasing star:** `fact_purchases` + `dim_vendor` (sharing `dim_product` with the sales star)
- **Sales star:** `fact_sales` + `dim_product` + `dim_sales_territory`

In [0]:
# GOLD — build fact_purchases: detail + header joined on PurchaseOrderID
# Pre-check: PurchaseOrderID must be unique in the header — otherwise the join would fan out
dup_po_ids = spark.sql("""
    SELECT COUNT(*) AS duplicate_po_ids
    FROM (
        SELECT PurchaseOrderID
        FROM adventure_works.silver.stg_purchase_order_header
        GROUP BY PurchaseOrderID
        HAVING COUNT(*) > 1
    )
""").collect()[0]["duplicate_po_ids"]
print(f"Duplicate PurchaseOrderIDs in header: {dup_po_ids} (0 = safe to join, no fan-out)")

spark.sql("""
CREATE OR REPLACE TABLE adventure_works.silver.fact_purchases AS
SELECT
    d.PurchaseOrderID,
    d.PurchaseOrderDetailID,
    d.DueDate,
    d.OrderQty,
    d.ProductID,
    d.UnitPrice,
    d.LineTotal,
    d.ReceivedQty,
    d.RejectedQty,
    d.StockedQty,
    h.VendorID,        -- pulled in from the header
    h.OrderDate,       -- pulled in from the header
    d.ModifiedDate
FROM adventure_works.silver.stg_purchase_order_detail d
JOIN adventure_works.silver.stg_purchase_order_header h
  ON d.PurchaseOrderID = h.PurchaseOrderID
""")

display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM adventure_works.silver.stg_purchase_order_detail) AS detail_rows,
    (SELECT COUNT(*) FROM adventure_works.silver.stg_purchase_order_header) AS header_rows,
    (SELECT COUNT(*) FROM adventure_works.silver.fact_purchases) AS fact_rows
"""))
# detail_rows = fact_rows proves the join kept one row per PO line (no fan-out)

Duplicate PurchaseOrderIDs in header: 0 (0 = safe to join, no fan-out)


detail_rows,header_rows,fact_rows
8845,4012,8845


## Query 1 — Thinnest Margin Between Sales Price and Purchase Price

Join `fact_sales` and `fact_purchases` on `ProductID` (both share `dim_product`).

A product can be sold and purchased at different prices over time, so compare the **average** sales price against the **average** purchase price per product. The thinnest margin = smallest difference (ordering ascending can even surface products sold below cost).

In [0]:
%sql
-- Top 5 products with the thinnest margin between sales price and purchase price
-- Average each price per product FIRST, then join: joining the raw facts directly
-- on ProductID would be many-to-many and distort the averages.
-- Zero prices (unsellable raw-material components) are excluded.
WITH sales_avg AS (
    SELECT ProductID, AVG(UnitPrice) AS avg_sales_price
    FROM adventure_works.default.fact_sales
    WHERE UnitPrice > 0
    GROUP BY ProductID
),
purchase_avg AS (
    SELECT ProductID, AVG(UnitPrice) AS avg_purchase_price
    FROM adventure_works.silver.fact_purchases
    WHERE UnitPrice > 0
    GROUP BY ProductID
)
SELECT
    p.Name AS product_name,
    ROUND(s.avg_sales_price, 4) AS avg_sales_price,
    ROUND(pu.avg_purchase_price, 4) AS avg_purchase_price,
    ROUND(s.avg_sales_price - pu.avg_purchase_price, 4) AS avg_margin
FROM sales_avg s
JOIN purchase_avg pu ON s.ProductID = pu.ProductID
JOIN adventure_works.default.dim_product p ON s.ProductID = p.ProductID
ORDER BY avg_margin ASC
LIMIT 5;

product_name,avg_sales_price,avg_purchase_price,avg_margin
Front Brakes,63.8760,82.8345,-18.9585
Rear Brakes,63.9000,82.8345,-18.9345
ML Road Tire,24.9900,39.4583,-14.4683
HL Mountain Pedal,48.5686,62.9895,-14.4209
Touring Pedal,48.5940,62.9895,-14.3955


## Query 2 — Units Sold vs. Units Purchased, by Month

Which products sell faster than we restock them?  
Aggregate units sold (`fact_sales`) and units purchased (`fact_purchases`) per product per month, join on both, and keep only months where sold > purchased (a restocking shortfall).

*Note: the sales month uses `ModifiedDate` — the true order date lives in `SalesOrderHeader`, which is not loaded in this lakehouse. Purchase month uses `OrderDate` from the header.*

In [0]:
%sql
-- Products that sell faster than we restock them, by month
-- FULL OUTER JOIN keeps the complete monthly comparison: months with sales only,
-- purchases only, or both. The flag keeps months where units_sold > units_purchased.
WITH sold AS (
    SELECT ProductID,
           date_trunc('month', ModifiedDate) AS month,
           SUM(OrderQty) AS units_sold
    FROM adventure_works.default.fact_sales
    GROUP BY 1, 2
),
purchased AS (
    SELECT ProductID,
           date_trunc('month', OrderDate) AS month,
           SUM(OrderQty) AS units_purchased
    FROM adventure_works.silver.fact_purchases
    GROUP BY 1, 2
)
SELECT
    p.Name AS product_name,
    COALESCE(s.month, pu.month) AS month,
    COALESCE(s.units_sold, 0) AS units_sold,
    COALESCE(pu.units_purchased, 0) AS units_purchased,
    COALESCE(s.units_sold, 0) - COALESCE(pu.units_purchased, 0) AS units_shortfall
FROM sold s
FULL OUTER JOIN purchased pu
  ON s.ProductID = pu.ProductID AND s.month = pu.month
JOIN adventure_works.default.dim_product p
  ON COALESCE(s.ProductID, pu.ProductID) = p.ProductID
WHERE COALESCE(s.units_sold, 0) > COALESCE(pu.units_purchased, 0)
ORDER BY units_shortfall DESC, units_sold DESC
LIMIT 20;

product_name,month,units_sold,units_purchased,units_shortfall
Water Bottle - 30 oz.,2025-03-01T00:00:00.000Z,673,0,673
Water Bottle - 30 oz.,2025-04-01T00:00:00.000Z,611,0,611
Water Bottle - 30 oz.,2024-07-01T00:00:00.000Z,603,0,603
Water Bottle - 30 oz.,2024-10-01T00:00:00.000Z,566,0,566
Water Bottle - 30 oz.,2024-09-01T00:00:00.000Z,552,0,552
AWC Logo Cap,2024-07-01T00:00:00.000Z,530,0,530
Water Bottle - 30 oz.,2025-02-01T00:00:00.000Z,519,0,519
Water Bottle - 30 oz.,2025-01-01T00:00:00.000Z,511,0,511
Water Bottle - 30 oz.,2024-08-01T00:00:00.000Z,508,0,508
"Classic Vest, S",2024-06-01T00:00:00.000Z,505,0,505


## Result — One Lakehouse, Two Stars Sharing dim_product

| Star | Fact | Dimensions |
| --- | --- | --- |
| **Sales** | `default.fact_sales` (121,317 rows) | `default.dim_product`, `default.dim_sales_territory` |
| **Purchasing** | `silver.fact_purchases` (8,845 rows) | `silver.dim_vendor` (+ shared `default.dim_product`) |

Medallion flow: **Bronze** raw staging (`_c0` names) → **Silver** named/typed/cleaned → **Gold** `fact_purchases` joined one-row-per-PO-line.